# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the [FAIR2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant metadata URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Inspect the record sets, their fields, and the corresponding `@id` values as described in the Croissant metadata.

We'll enumerate all record sets and their fields using their `@id`. These are required to reference entities with `mlcroissant`.

In [ ]:
from pprint import pprint

# Show list of record sets with their @id and name
print("Record Sets (with @id):")
for rs in metadata.record_sets:
    print(f"  @id: {rs.id}, name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

## 3. Data Extraction
Let's extract data from a record set of interest. We'll use the `@id`s from the overview above.

We'll load all record sets into Pandas DataFrames for easy analysis.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")

# Choose a record set for demonstration
main_record_set_id = record_set_ids[0]
print(f"Columns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's:
- Filter numeric fields
- Normalize selected fields
- Group and aggregate data

All columns are referenced by their `@id`.

You can reference the output from Section 2 and 3 for the exact `@id`s of fields.

In [ ]:
# Identify a numeric field (by @id) from the printed columns
# For this example, let's assume one field is 'protein_abundance' (change to your data)
numeric_field_id = None
for field in metadata.record_sets[0].fields:
    if field.data_type in ['Float', 'Integer', 'Number']:
        numeric_field_id = field.id
        print(f"Using numeric field: {numeric_field_id} ({field.name})")
        break
if numeric_field_id is None:
    print("No numeric field found; please adjust field selection for your dataset.")

# Prepare DataFrame
df = dataframes[main_record_set_id]

# Drop rows with missing values in the numeric field
filtered_df = df.dropna(subset=[numeric_field_id])
threshold = filtered_df[numeric_field_id].mean() if numeric_field_id else 0
if numeric_field_id:
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
if numeric_field_id:
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by the first non-numeric field
group_field_id = None
for field in metadata.record_sets[0].fields:
    if field.data_type == 'Text':
        group_field_id = field.id
        print(f"Grouping by field: {group_field_id} ({field.name})")
        break
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped and averaged {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field and group means using matplotlib.

You may choose a different field by adjusting the variable assignment above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the normalized field
if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Bar plot for group means
if group_field_id and numeric_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,4))
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    group_means.sort_values(ascending=False).plot(kind='bar')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We successfully explored the FAIR2 mass spectrometry dataset using the `mlcroissant` library.
- We identified record sets and fields using their `@id` as required by the Croissant schema.
- Basic data processing was conducted, including filtering and normalizing a numeric field and grouping by a textual attribute.
- Data distribution and group-wise means were visualized.

Further analyses can be built by selecting specific record sets and experimenting with the rich field structure described in the Croissant schema.
